# Show the WidowX AI in the MuJoCo simulation

Move the simulated WidowX AI (`src/robot/wxai`, driver `mujoco`) through a few Cartesian waypoints, opening and closing the gripper, and watch it through the cameras of the scene: the fixed `front` and `side` cameras and the `cam` wrist camera of the arm. The images update live while the arm moves; at the end the movement is replayed as an animated GIF.

Nothing is saved: the exprun is resolved without creating its result directory. See `src/robot/wxai/PLAN-WXAI.md` and `INSTALL-WXAI.md`.

In [ ]:
import sys
sys.path.append("..")

import io
from copy import copy

import numpy as np
from IPython.display import Image, display
from PIL import Image as PILImage

from exp_run_config import Config
from robot.wxai import PositionController, WXAICommand
from robot.wxai.simulation.mujoco_runtime import get_runtime

Config.PROJECTNAME = "BerryPicker"

In [ ]:
# the cameras of the scene, side by side, and their size
cameras = ["front", "side", "cam"]
width, height = 320, 240
# simulation time per frame in seconds, and the time of a move between two waypoints
tick = 0.05
moving_time = 1.5

exp = Config().get_experiment("robot_wxai", "position_controller_mujoco_wxai_00", create_data_dir=False)

In [ ]:
rob = PositionController(exp)
rob.start_robot()
runtime = get_runtime(exp["scene"])

def frame():
    """The camera images side by side, as an RGB array."""
    images = [runtime.render(camera, width, height)[:, :, ::-1] for camera in cameras]
    return np.hstack(images)

def png(array):
    buffer = io.BytesIO()
    PILImage.fromarray(array).save(buffer, format="PNG")
    return buffer.getvalue()

rob.go_home(moving_time=2.0)
home = rob.get_position()
print(home)

In [ ]:
# waypoints relative to the home pose (meters and radians), and the gripper action at each
waypoints = [
    ({"x": -0.10, "z": -0.15}, "release"),
    ({"x": -0.10, "y": 0.15, "z": -0.25, "pitch": 0.6}, "grasp"),
    ({"x": -0.15, "y": -0.15, "z": -0.20, "pitch": 0.4, "yaw": -0.5}, "release"),
    ({"z": -0.10, "roll": 0.5}, "grasp"),
    ({}, "hold"),
]

frames = []
view = display(Image(data=png(frame())), display_id=True)
for offsets, gripper_action in waypoints:
    target = copy(home)
    for field, offset in offsets.items():
        target[field] += offset
    if not rob.can_reach(target):
        print(f"Skipping an unreachable waypoint:\n{target}")
        continue
    rob.move(WXAICommand(target, gripper_action), moving_time=moving_time, blocking=False)
    for _ in range(round((moving_time + 0.5) / tick)):
        rob.update(tick)
        frames.append(frame())
        view.update(Image(data=png(frames[-1])))
    error = max(abs(rob.get_position()[field] - target[field]) for field in ("x", "y", "z"))
    print(f"Reached waypoint {offsets} with {gripper_action}, position error {error:.4f} m")

In [ ]:
rob.stop_robot()
print(f"Recorded {len(frames)} frames of {tick} s simulation time")

The whole movement as an animated GIF, at the speed of the simulation.

In [ ]:
buffer = io.BytesIO()
images = [PILImage.fromarray(array) for array in frames[::2]]
images[0].save(buffer, format="GIF", save_all=True, append_images=images[1:],
               duration=int(2 * tick * 1000), loop=0)
display(Image(data=buffer.getvalue(), format="gif"))